# Trasmissione: riferimenti senza membrana e rapporto con le membrane

**Parte 1 - acquisizione dei riferimenti (celle 1-6, con gli strumenti).** Scrivi le coordinate di **2 punti dove non ci sono membrane** (`RIF_1`, `RIF_2` in cella 1). Per ciascuno il notebook muove i motori X, Y e acquisisce con l'oscilloscopio:
- `CON_SCANSIONE = True`: **con la scansione del laser** (stessa sequenza degli altri notebook di acquisizione): si ottiene lo spettro di riferimento in funzione della rampa di CH1;
- `CON_SCANSIONE = False`: **senza scansione** (laser fermo, scope in single con trigger forzato): si ottiene il livello di riferimento.

`N_ACQUISIZIONI` ripete l'acquisizione su ogni punto (le ripetizioni vengono mediate). I file sono salvati in `AAAA-MM-GG/rif_NNN/` con un registro `riferimenti.csv`.

**Parte 2 - trasmissione (celle 7-10, solo analisi, senza strumenti).** Scrivi la cartella dei riferimenti (`CARTELLA_RIFERIMENTO`) e la cartella del run di membrane (`CARTELLA_RUN`) oppure un singolo CSV (`PERCORSO_CSV_MEMBRANA`). La trasmissione è

**T = (S_membrana − S_buio) / (S_riferimento − S_buio)**

con S = CH2. Con la scansione, ogni acquisizione è ridotta alla sua rampa di CH1 (mediana a blocchi: toglie i picchi spuri) e **allineata in tensione di CH1**, per cui piccole differenze di tempistica o di durata della rampa tra le acquisizioni non contano; T è una curva in funzione di CH1 (o della lunghezza d'onda con `ASSE_X = "lunghezza_onda"`, ipotesi 0-10 V ↔ 1520-1570 nm da verificare). Senza scansione T è un numero (rapporto delle mediane di CH2).

Il riferimento usato per una membrana è la **media** dei due punti (`RIFERIMENTO = "media"`) oppure l'**interpolazione lineare** tra i due in base alla posizione della membrana lungo il segmento che li unisce (`"interpolato"`, utile se la potenza varia lungo il campione). La cella 8 stampa di quanto differiscono i due riferimenti: se differiscono molto, la trasmissione ha un'incertezza di quell'ordine.

Ordine: celle 1-4 una volta; la cella 5 mostra l'anteprima (**non muove niente**); la cella 6 acquisisce i riferimenti e chiede conferma. Per sola analisi, salta alla cella 7 (le celle 1-6 non servono).

In [ ]:
# Cella 1 - configurazione e funzioni (scope, laser, griglia)
import os
import sys
import time
import datetime

import numpy as np
import pyvisa

OSC_VISA = "USB::0x0699::0x0408::C028813::INSTR"
OSC_TIMEOUT_MS = 30000

CHANNELS = ["CH1", "CH2"]

LASER_WAVELENGTH_START = 1520.0  # nm, come trigger_scan.m
LASER_WAVELENGTH_STOP = 1570.0   # nm, come trigger_scan.m

POLL_INTERVAL_S = 0.1
CHANNEL_READ_PAUSE_S = 0.1

# ---------------- punti di riferimento SENZA membrana (in mm, limiti motori 0-150) ----------------
RIF_1 = None               # <-- SCRIVI QUI (X, Y) del primo punto senza membrana, es. (91.50, 146.20)
RIF_2 = None               # <-- SCRIVI QUI (X, Y) del secondo punto senza membrana
CON_SCANSIONE = True       # True: con la scansione del laser (spettro di riferimento); False: senza scansione (laser fermo)
N_ACQUISIZIONI = 1         # acquisizioni per punto (le ripetizioni vengono mediate nell'analisi)

ASSESTAMENTO_S = 1.0           # pausa dopo il movimento, prima di acquisire
ATTESA_RITORNO_LASER_S = 5.0  # tra due scan, perché il laser torni a 1520 nm (CH1 basso)
RESETTA_CONTATORE_SCAN = True  # prima di ogni start: OUTP:SCAN:STOP + SOUR:WAVE:DESSCANS 1 (il laser potrebbe smettere di partire dopo N scan)
# --------------------------------------------------------------------------


def crea_cartella_run():
    oggi = datetime.date.today().strftime("%Y-%m-%d")
    base_folder = os.path.join(os.getcwd(), oggi)
    os.makedirs(base_folder, exist_ok=True)
    run_esistenti = [d for d in os.listdir(base_folder) if d.startswith("run_")]
    run_folder = os.path.join(base_folder, f"run_{len(run_esistenti) + 1:03d}")
    os.makedirs(run_folder)
    return run_folder


def configura_scope(osc):
    osc.write("HORizontal:MODE MANUAL")
    rec_len = int(osc.query("HORizontal:RECORDLENGTH?"))
    osc.write("DATA:START 1")
    osc.write(f"DATA:STOP {rec_len}")
    osc.write("DATA:WIDTH 1")
    osc.write("DATA:ENCdg ASCii")


def arm_acquisition(osc, timeout_s=120):
    osc.write("ACQ:STOPAFTER SEQUENCE")
    osc.write("ACQ:STATE RUN")
    # Lo scan parte solo quando lo scope e' READY (finito il pre-trigger).
    print("Attendo che lo scope finisca il pre-trigger (-> READY)...")
    t0 = time.time()
    ultimo = None
    while True:
        stato = osc.query("TRIGger:STATE?").strip().upper()
        if stato != ultimo:
            print(f"  stato trigger: {stato}")
            ultimo = stato
        if stato == "READY":
            break
        if time.time() - t0 > timeout_s:
            raise TimeoutError(
                f"Lo scope non e' arrivato a READY in {timeout_s} s "
                f"(stato trigger: {stato}, ACQ:STATE: {osc.query('ACQ:STATE?').strip()})"
            )
        time.sleep(POLL_INTERVAL_S)
    print("Scope pronto ad accettare il trigger.")


def wait_for_acquisition(osc):
    print("Attendo che l'oscilloscopio completi l'acquisizione triggerata...")
    while True:
        if int(osc.query("ACQ:STATE?")) == 0:
            print("Acquisizione completata!")
            break
        time.sleep(POLL_INTERVAL_S)


def acquire_channel(osc, source="CH1"):
    osc.write(f"DATA:SOURCE {source}")
    ymult = float(osc.query("WFMPRE:YMULT?"))
    yoff = float(osc.query("WFMPRE:YOFF?"))
    yzero = float(osc.query("WFMPRE:YZERO?"))
    print(f"{source}: YMULT={ymult}, YOFF={yoff}, YZERO={yzero}")
    raw = osc.query("CURVE?")
    ydata = np.array(raw.split(","), dtype=float)
    return (ydata - yoff) * ymult + yzero

In [ ]:
# Cella 2 - connessione all'oscilloscopio (una volta per sessione)
rm = pyvisa.ResourceManager()
osc = rm.open_resource(OSC_VISA)
osc.timeout = OSC_TIMEOUT_MS
configura_scope(osc)
print("Oscilloscopio connesso e configurato.")

In [ ]:
# Cella 3 - laser: carica la DLL e apre il dispositivo (una volta per sessione)
USBADDR = 1
DEVICE_ID = 0x100A  # hex2dec('100A')

dll_path = os.path.abspath(os.path.join(os.getcwd(), "..", "..", "scripts", "laser", "UsbDllWrap.dll"))
if not os.path.isfile(dll_path):
    raise RuntimeError(f"UsbDllWrap.dll non trovata in {dll_path} (il notebook va aperto da Chiara_Bru/runs_oscilloscope)")
print(f"DLL trovata: {dll_path}")
sys.path.append(os.path.dirname(dll_path))

import clr
clr.AddReference("UsbDllWrap")
import Newport
from System.Text import StringBuilder

NP_USB = Newport.USBComm.USB()

esito = NP_USB.OpenDevices(DEVICE_ID)
print(f"OpenDevices({hex(DEVICE_ID)}) -> {esito}")
querydata = StringBuilder(64)


def laser_query(comando):
    esito = NP_USB.Query(USBADDR, comando, querydata)
    risposta = querydata.ToString()
    print(f"Query {comando!r} -> esito={esito}, risposta={risposta!r}")
    return risposta


LASER_COMANDI_CON_QUERY = True  # True: come i driver di riferimento (Query + lettura dell'OK)


def laser_write(comando):
    if LASER_COMANDI_CON_QUERY:
        # Con Write la risposta "OK" del laser resta non letta e si accumula,
        # sfasando le query successive e saturando la coda del laser.
        esito = NP_USB.Query(USBADDR, comando, querydata)
        print(f"Write {comando!r} -> esito={esito}, risposta={querydata.ToString()!r}")
    else:
        esito = NP_USB.Write(USBADDR, comando)
        print(f"Write {comando!r} -> esito={esito}")


laser_query("*IDN?")
laser_query("OUTP:STAT?")

In [ ]:
# Cella 4 - motori PI C-863: X = ASRL4, Y = ASRL5 (una volta per sessione)
MOTORE_X = "ASRL4::INSTR"
MOTORE_Y = "ASRL5::INSTR"
PI_BAUD = 19200
PI_AXIS = "1"
PI_TIMEOUT_MS = 2000
PI_TIMEOUT_MOVIMENTO_S = 60
PI_TOLLERANZA_MM = 0.01  # valore usato nella prova sul laboratorio (i motori arrivano entro circa 10 um)


def pi_apri(porta):
    inst = rm.open_resource(porta)
    inst.baud_rate = PI_BAUD
    inst.timeout = PI_TIMEOUT_MS
    inst.read_termination = "\n"
    inst.write_termination = "\n"
    return inst


def pi_query(inst, comando):
    return inst.query(comando).strip()


def pi_valore(inst, comando):
    """Estrae il numero da risposte tipo '1=+0012.3456'."""
    return float(pi_query(inst, comando).split("=")[-1])


def pi_limiti(inst):
    return pi_valore(inst, f"TMN? {PI_AXIS}"), pi_valore(inst, f"TMX? {PI_AXIS}")


def pi_muovi(inst, nome, target):
    """Movimento assoluto in mm, con controllo dei limiti prima di muovere."""
    tmin, tmax = pi_limiti(inst)
    if not (tmin <= target <= tmax):
        raise ValueError(f"Motore {nome}: target {target} mm fuori dai limiti {tmin}..{tmax} mm")
    inst.write(f"MOV {PI_AXIS} {target:.4f}")
    time.sleep(0.1)
    t0 = time.time()
    while True:
        in_target = pi_query(inst, f"ONT? {PI_AXIS}").endswith("1")
        pos = pi_valore(inst, f"POS? {PI_AXIS}")
        if in_target and abs(pos - target) <= PI_TOLLERANZA_MM:
            break
        if time.time() - t0 > PI_TIMEOUT_MOVIMENTO_S:
            raise TimeoutError(f"Motore {nome}: non arrivato a {target} mm (posizione {pos} mm)")
        time.sleep(0.1)
    errore = int(pi_query(inst, "ERR?"))
    if errore != 0:
        raise RuntimeError(f"Motore {nome}: ERR? = {errore} dopo il movimento a {target} mm")
    print(f"Motore {nome}: a {pos:.4f} mm")


motore_x = pi_apri(MOTORE_X)
motore_y = pi_apri(MOTORE_Y)

for nome, inst in (("X", motore_x), ("Y", motore_y)):
    errore_residuo = pi_query(inst, "ERR?")  # legge e azzera un eventuale errore vecchio
    tmin, tmax = pi_limiti(inst)
    print(f"Motore {nome}: {pi_query(inst, '*IDN?')}")
    print(f"   posizione {pi_valore(inst, f'POS? {PI_AXIS}')} mm | limiti {tmin}..{tmax} mm | "
          f"servo {pi_query(inst, f'SVO? {PI_AXIS}')} | referenziato {pi_query(inst, f'FRF? {PI_AXIS}')} | "
          f"errore residuo letto: {errore_residuo}")

In [ ]:
# Cella 5 - punti di riferimento: ANTEPRIMA (NON muove niente)
if RIF_1 is None or RIF_2 is None:
    raise ValueError("Scrivi in cella 1 le coordinate (X, Y) di RIF_1 e RIF_2 (punti SENZA membrana) e rilancia la cella 1")

PUNTI = [(float(RIF_1[0]), float(RIF_1[1])), (float(RIF_2[0]), float(RIF_2[1]))]
for nome, inst, k in (("X", motore_x, 0), ("Y", motore_y, 1)):
    tmin, tmax = pi_limiti(inst)
    for n, p in enumerate(PUNTI, start=1):
        if not (tmin <= p[k] <= tmax):
            raise ValueError(f"Riferimento {n}: {nome} = {p[k]} mm fuori dai limiti {tmin}..{tmax} mm")

print("Riferimenti (punti SENZA membrana):")
for n, (x, y) in enumerate(PUNTI, start=1):
    print(f"  {n}: X={x:.4f}  Y={y:.4f}")
print(f"Modalita': {'CON la scansione del laser' if CON_SCANSIONE else 'SENZA scansione (laser fermo, trigger forzato)'} | {int(N_ACQUISIZIONI)} acquisizione/i per punto")
print(f"Spazio su disco stimato: circa {len(PUNTI) * int(N_ACQUISIZIONI) * 0.24:.2f} GB")

In [ ]:
# Cella 6 - ACQUISISCE i riferimenti: per ogni punto muove X e Y e acquisisce (con o senza scansione)
import csv

if input(f"Muovere i motori e acquisire {len(PUNTI) * int(N_ACQUISIZIONI)} riferimenti? scrivi 'si' per confermare: ").strip().lower() != "si":
    raise RuntimeError("Annullato: nessun movimento eseguito.")

oggi = datetime.date.today().strftime("%Y-%m-%d")
base_folder = os.path.join(os.getcwd(), oggi)
os.makedirs(base_folder, exist_ok=True)
esistenti = [d for d in os.listdir(base_folder) if d.startswith("rif_")]
rif_folder = os.path.join(base_folder, f"rif_{len(esistenti) + 1:03d}")
os.makedirs(rif_folder)
print(f"I riferimenti saranno salvati in: {rif_folder}")

registro_rif = os.path.join(rif_folder, "riferimenti.csv")
with open(registro_rif, "w", newline="") as f:
    csv.writer(f).writerow(["indice", "x_mm", "y_mm", "file", "con_scansione"])

try:
    contatore = 0
    for n, (x, y) in enumerate(PUNTI, start=1):
        print(f"\n=== Riferimento {n}/{len(PUNTI)}: X={x:.4f} mm, Y={y:.4f} mm ===")
        pi_muovi(motore_x, "X", x)
        pi_muovi(motore_y, "Y", y)
        time.sleep(ASSESTAMENTO_S)

        for rip in range(int(N_ACQUISIZIONI)):
            if CON_SCANSIONE:
                laser_write(f"SOUR:WAVE:START {LASER_WAVELENGTH_START}")
                laser_write(f"SOUR:WAVE:STOP {LASER_WAVELENGTH_STOP}")
                if RESETTA_CONTATORE_SCAN:
                    laser_write("OUTP:SCAN:STOP")
                    laser_write("SOUR:WAVE:DESSCANS 1")
                arm_acquisition(osc)
                laser_write("OUTP:SCAN:START")
            else:
                arm_acquisition(osc)
                osc.write("TRIGger FORCe")          # nessuna scansione: il trigger viene forzato
            wait_for_acquisition(osc)
            time.sleep(1.0)

            data = {}
            for ch in CHANNELS:
                print(f"Leggo {ch}...")
                data[ch] = acquire_channel(osc, ch)
                time.sleep(CHANNEL_READ_PAUSE_S)

            nome_file = f"riferimento_{n}_rip{rip + 1}_x{x:.3f}_y{y:.3f}.csv"
            min_len = min(len(v) for v in data.values())
            colonne = np.column_stack([data[ch][:min_len] for ch in CHANNELS])
            np.savetxt(os.path.join(rif_folder, nome_file), colonne, delimiter=",", header=",".join(CHANNELS), comments="")
            with open(registro_rif, "a", newline="") as f:
                csv.writer(f).writerow([contatore, f"{x:.4f}", f"{y:.4f}", nome_file, CON_SCANSIONE])
            contatore += 1
            print(f"Salvato: {nome_file}")

            if CON_SCANSIONE and (rip < int(N_ACQUISIZIONI) - 1 or n < len(PUNTI)):
                print(f"Attendo {ATTESA_RITORNO_LASER_S:.0f} s che il laser torni a {LASER_WAVELENGTH_START} nm...")
                time.sleep(ATTESA_RITORNO_LASER_S)
except BaseException:
    if CON_SCANSIONE:
        print("\nInterrotto: provo a fermare lo scan del laser.")
        try:
            laser_write("OUTP:SCAN:STOP")
        except Exception as e:
            print(f"Impossibile fermare lo scan: {e}")
    raise

print(f"\nRiferimenti acquisiti: {contatore} file in {rif_folder}")
print("Per l'analisi scrivi in cella 7:  CARTELLA_RIFERIMENTO = " + repr(os.path.relpath(rif_folder, os.getcwd()).replace(os.sep, "/")))

In [ ]:
# Cella 7 - ANALISI (non servono gli strumenti): impostazioni e funzioni
import csv
import os

import numpy as np
import matplotlib.pyplot as plt

try:
    import pandas as pd
except ImportError:
    pd = None

CARTELLA_RIFERIMENTO = ""      # <-- SCRIVI QUI la cartella dei riferimenti (con riferimenti.csv), es. "2026-10-12/rif_001"
CARTELLA_RUN = ""              # cartella di un run di membrane (con griglia.csv), es. "2026-10-06/run_007"
PERCORSO_CSV_MEMBRANA = ""     # in alternativa: un singolo CSV di una membrana (ha la precedenza su CARTELLA_RUN per la cella 9)
PUNTO = 0                      # per CARTELLA_RUN: indice del punto mostrato nella cella 9

RIFERIMENTO = "media"          # "media" dei due punti | "interpolato" in base alla posizione della membrana lungo il segmento tra i due
LIVELLO_BUIO_V = 0.0           # livello di CH2 senza luce (offset dello scope/rivelatore), sottratto da membrana e riferimento
ASSE_X = "tensione_ch1"        # "tensione_ch1" | "lunghezza_onda" (conversione con V_0, V_1, LAMBDA_0, LAMBDA_1: ipotesi da verificare)
V_0, V_1 = 0.0, 10.0
LAMBDA_0, LAMBDA_1 = 1520.0, 1570.0

CANALE = "CH2"
CANALE_RAMPA = "CH1"
V_INIZIO_V = 0.9               # rampa di CH1: inizio quando CH1 supera questo valore
TOLLERANZA_FINE_V = 0.3        # rampa di CH1: fine quando CH1 e' a meno di questo dal suo massimo
N_PUNTI_CURVA = 5000           # ogni acquisizione e' ridotta a circa questo numero di punti (mediana a blocchi)
N_GRIGLIA = 2000               # punti della griglia comune su cui si calcola T
SOGLIA_RIF = 0.02              # dove S_riferimento - buio e' sotto questa frazione del suo massimo, T non e' definita (NaN)


def leggi_csv(percorso):
    if pd is not None:
        df = pd.read_csv(percorso, dtype=np.float32)
        ch1 = df[CANALE_RAMPA].to_numpy(dtype=float) if CANALE_RAMPA in df.columns else None
        return ch1, df[CANALE].to_numpy(dtype=float)
    dati = np.loadtxt(percorso, delimiter=",", skiprows=1)
    return dati[:, 0].astype(float), dati[:, 1].astype(float)


def rampa_ch1(ch1):
    if ch1 is None:
        return None
    sopra = np.flatnonzero(ch1 > V_INIZIO_V)
    if sopra.size == 0:
        return None
    i0 = int(sopra[0])
    i1 = i0 + int(np.flatnonzero(ch1[i0:] >= ch1.max() - TOLLERANZA_FINE_V)[0])
    return (i0, i1) if i1 > i0 + 10 else None


def curva_su_rampa(ch1, ch2):
    """CH2 nella rampa di CH1 come (v, s): v = tensione di CH1 (retta fittata sulla rampa) al centro di ogni blocco,
    s = mediana di CH2 nel blocco (i picchi spuri non contano). None se non c'e' una rampa."""
    rampa = rampa_ch1(ch1)
    if rampa is None:
        return None
    i0, i1 = rampa
    idx = np.arange(i0, i1 + 1)
    m, c0 = np.polyfit(idx, ch1[i0:i1 + 1], 1)       # CH1 e' quantizzato: si usa la retta fittata
    if m <= 0:
        return None
    n = max(1, (i1 - i0 + 1) // int(N_PUNTI_CURVA))
    k = (i1 - i0 + 1) // n
    s = np.median(ch2[i0:i0 + k * n].reshape(k, n), axis=1)
    centri = i0 + (np.arange(k) + 0.5) * n - 0.5
    return m * centri + c0, s


def asse_x(v):
    if ASSE_X == "tensione_ch1":
        return v, "CH1 (V)"
    if ASSE_X == "lunghezza_onda":
        return LAMBDA_0 + (v - V_0) / (V_1 - V_0) * (LAMBDA_1 - LAMBDA_0), "lunghezza d'onda (nm)"
    raise ValueError("ASSE_X deve essere 'tensione_ch1' o 'lunghezza_onda'")

In [ ]:
# Cella 8 - carica i riferimenti (2 punti senza membrana, eventuali ripetizioni mediate)
if not CARTELLA_RIFERIMENTO.strip():
    raise ValueError("Scrivi in cella 7 la cartella dei riferimenti in CARTELLA_RIFERIMENTO")
cartella_rif = os.path.abspath(CARTELLA_RIFERIMENTO)
with open(os.path.join(cartella_rif, "riferimenti.csv"), newline="") as f:
    righe_rif = list(csv.DictReader(f))

gruppi = {}                                  # (x, y) -> lista di (ch1, ch2)
for r in righe_rif:
    gruppi.setdefault((round(float(r["x_mm"]), 4), round(float(r["y_mm"]), 4)), []).append(
        leggi_csv(os.path.join(cartella_rif, r["file"])))
if len(gruppi) != 2:
    print(f"Attenzione: ci sono {len(gruppi)} punti di riferimento distinti (ne servono 2)")

# modalita': con scansione se in tutte le acquisizioni c'e' una rampa di CH1
curve = {pos: [curva_su_rampa(c1, c2) for c1, c2 in lista] for pos, lista in gruppi.items()}
con_rampa = [c is not None for lista in curve.values() for c in lista]
if all(con_rampa):
    modo = "scansione"
elif not any(con_rampa):
    modo = "livello"
else:
    raise RuntimeError("I riferimenti sono misti: alcuni con rampa di CH1 e altri senza")
print(f"Modalita' dei riferimenti: {'CON scansione (spettro)' if modo == 'scansione' else 'SENZA scansione (livello)'}")

rif = {"modo": modo, "gruppi": []}
if modo == "scansione":
    v_lo = max(c[0][0] for lista in curve.values() for c in lista)
    v_hi = min(c[0][-1] for lista in curve.values() for c in lista)
    griglia_v = np.linspace(v_lo, v_hi, int(N_GRIGLIA))
    rif["griglia_v"] = griglia_v
    for pos, lista in curve.items():
        s = np.mean([np.interp(griglia_v, v, s) for v, s in lista], axis=0)      # media delle ripetizioni
        rif["gruppi"].append({"pos": pos, "S": s})
    print(f"Griglia comune: CH1 da {v_lo:.3f} a {v_hi:.3f} V ({int(N_GRIGLIA)} punti)")
else:
    for pos, lista in gruppi.items():
        rif["gruppi"].append({"pos": pos, "S": float(np.mean([np.median(c2) for _, c2 in lista]))})

# controllo di riproducibilita' dei due riferimenti
if len(rif["gruppi"]) == 2:
    S1, S2 = rif["gruppi"][0]["S"] - LIVELLO_BUIO_V, rif["gruppi"][1]["S"] - LIVELLO_BUIO_V
    if modo == "scansione":
        buono = np.abs(S1) > SOGLIA_RIF * np.abs(S1).max()
        rapporto = S2[buono] / S1[buono]
        print(f"Rapporto tra i due riferimenti (2/1): media {rapporto.mean():.4f}, deviazione standard {rapporto.std():.4f}, "
              f"min {rapporto.min():.4f}, max {rapporto.max():.4f}")
        x_plot, etichetta = asse_x(rif["griglia_v"])
        fig, (a1, a2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
        for g, nome in zip(rif["gruppi"], ("riferimento 1", "riferimento 2")):
            a1.plot(x_plot, g["S"], label=f"{nome} (X={g['pos'][0]}, Y={g['pos'][1]})")
        a1.set_ylabel(f"{CANALE} (V)")
        a1.legend(loc="best")
        a2.plot(x_plot[buono], rapporto, color="k")
        a2.set_ylabel("rif. 2 / rif. 1")
        a2.set_xlabel(etichetta)
        plt.tight_layout()
        plt.show()
    else:
        print(f"Livelli dei riferimenti: {S1 + LIVELLO_BUIO_V:.5g} V e {S2 + LIVELLO_BUIO_V:.5g} V (rapporto 2/1 = {S2 / S1:.4f})")

In [ ]:
# Cella 9 - trasmissione di UNA membrana (CSV singolo o un punto di un run)
def s_riferimento(posizione):
    """Riferimento (curva sulla griglia o livello) per una membrana a 'posizione' = (x, y) o None."""
    gr = rif["gruppi"]
    if len(gr) == 1 or RIFERIMENTO == "media" or posizione is None:
        return np.mean([g["S"] for g in gr], axis=0)
    p1, p2 = np.array(gr[0]["pos"]), np.array(gr[1]["pos"])
    w = float(np.clip(np.dot(np.array(posizione) - p1, p2 - p1) / np.dot(p2 - p1, p2 - p1), 0.0, 1.0))
    return (1 - w) * gr[0]["S"] + w * gr[1]["S"]


def trasmissione_membrana(ch1, ch2, posizione):
    """Restituisce (T, S_membrana, S_riferimento): curve sulla griglia comune (scansione) o numeri (livello)."""
    sref = s_riferimento(posizione) - LIVELLO_BUIO_V
    if rif["modo"] == "scansione":
        c = curva_su_rampa(ch1, ch2)
        if c is None:
            raise RuntimeError("Questa membrana non ha una rampa di CH1 ma i riferimenti si': misura incoerente")
        smem = np.interp(rif["griglia_v"], c[0], c[1], left=np.nan, right=np.nan) - LIVELLO_BUIO_V
        T = np.where(np.abs(sref) > SOGLIA_RIF * np.abs(sref).max(), smem / sref, np.nan)
        return T, smem + LIVELLO_BUIO_V, sref + LIVELLO_BUIO_V
    smem = float(np.median(ch2)) - LIVELLO_BUIO_V
    return smem / sref, smem + LIVELLO_BUIO_V, sref + LIVELLO_BUIO_V


if PERCORSO_CSV_MEMBRANA.strip():
    percorso_m = os.path.abspath(PERCORSO_CSV_MEMBRANA)
    posizione = None
    titolo = os.path.basename(percorso_m)
elif CARTELLA_RUN.strip():
    cartella_run = os.path.abspath(CARTELLA_RUN)
    with open(os.path.join(cartella_run, "griglia.csv"), newline="") as f:
        registro = list(csv.DictReader(f))
    r = next((r for r in registro if int(r["indice"]) == PUNTO), None)
    if r is None:
        raise ValueError(f"Punto {PUNTO} non presente nel registro")
    percorso_m = os.path.join(cartella_run, r["file"])
    posizione = (float(r["x_mm"]), float(r["y_mm"]))
    titolo = f"punto {PUNTO} (X={r['x_mm']}, Y={r['y_mm']})"
else:
    raise ValueError("Scrivi in cella 7 PERCORSO_CSV_MEMBRANA oppure CARTELLA_RUN")

ch1_m, ch2_m = leggi_csv(percorso_m)
T, smem, sref = trasmissione_membrana(ch1_m, ch2_m, posizione)

if rif["modo"] == "scansione":
    x_plot, etichetta = asse_x(rif["griglia_v"])
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
    a1.plot(x_plot, sref, label="riferimento")
    a1.plot(x_plot, smem, label="membrana")
    a1.set_ylabel(f"{CANALE} (V)")
    a1.legend(loc="best")
    a2.plot(x_plot, T, color="k")
    a2.axhline(1, color="gray", lw=0.8, ls="--")
    a2.set_ylabel("trasmissione T")
    a2.set_xlabel(etichetta)
    a1.set_title(titolo)
    plt.tight_layout()
    plt.show()
    print(f"T: media {np.nanmean(T):.4f}, minimo {np.nanmin(T):.4f}, massimo {np.nanmax(T):.4f}")
else:
    print(f"{titolo}: S_membrana = {smem:.5g} V, S_riferimento = {sref:.5g} V -> T = {T:.5f}")

In [ ]:
# Cella 10 - trasmissione di TUTTE le membrane del run (riassunto in trasmissione_valori.csv, curve in trasmissione_curve.npz)
if not CARTELLA_RUN.strip():
    raise ValueError("Scrivi in cella 7 CARTELLA_RUN per calcolare la trasmissione di tutto il run")
cartella_run = os.path.abspath(CARTELLA_RUN)
with open(os.path.join(cartella_run, "griglia.csv"), newline="") as f:
    registro = list(csv.DictReader(f))

righe_out, curve_T = [], []
for r in registro:
    percorso = os.path.join(cartella_run, r["file"])
    if not os.path.isfile(percorso):
        print(f"  punto {r['indice']}: file mancante, salto")
        continue
    ch1_m, ch2_m = leggi_csv(percorso)
    T, _, _ = trasmissione_membrana(ch1_m, ch2_m, (float(r["x_mm"]), float(r["y_mm"])))
    if rif["modo"] == "scansione":
        righe_out.append([int(r["indice"]), r["x_mm"], r["y_mm"], np.nanmean(T), np.nanmin(T), np.nanmax(T)])
        curve_T.append(T)
    else:
        righe_out.append([int(r["indice"]), r["x_mm"], r["y_mm"], T, T, T])
    print(f"  punto {r['indice']}: T media = {righe_out[-1][3]:.4f}, min = {righe_out[-1][4]:.4f}")

with open(os.path.join(cartella_run, "trasmissione_valori.csv"), "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["indice", "x_mm", "y_mm", "T_media", "T_min", "T_max"])
    w.writerows(righe_out)
if curve_T:
    np.savez(os.path.join(cartella_run, "trasmissione_curve.npz"), v_ch1=rif["griglia_v"], T=np.array(curve_T),
             indici=np.array([r[0] for r in righe_out]))
print(f"Salvato in {cartella_run}: trasmissione_valori.csv" + (" e trasmissione_curve.npz" if curve_T else ""))